## Crossover sensitivity

**Calculate percentage of patients that are high risk at different crossover thresholds: 14d, 30d, and 60dd**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate
327,FDD3D7DBFEA18,pembro_platinum,2021-11-30
642,FEB7F24503164,pembro_platinum,2020-11-02
1224,FC0EED9A68984,pembro,2023-04-24


In [4]:
treatment_df.shape

(1854, 3)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'pembro_platinum').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [7]:
features_df.shape

(1736, 162)

In [8]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [9]:
surv_pred_df.head(3)

,PatientID,psurv_180_calibrated
0,FEBCC4F65CC4C,0.981994
1,F2C55497C35F5,0.076107
2,F51465980052A,0.926062


In [10]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [11]:
df.shape

(1736, 165)

In [12]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [13]:
df.shape

(1736, 166)

In [14]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [15]:
df['treatment_year'] = df['StartDate'].dt.year

In [16]:
df = df.query('treatment_year <= 2022')

In [17]:
df.shape

(1480, 167)

In [18]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate_30 = float(f.read())

with open('../outputs/crossover_survival_estimate_14.txt', 'r') as f:
    crossover_survival_estimate_14 = float(f.read())

with open('../outputs/crossover_survival_estimate_60.txt', 'r') as f:
    crossover_survival_estimate_60 = float(f.read())

In [19]:
print(f'r* for 14d crossover: {crossover_survival_estimate_14}')
print(f'r* for 30d crossover: {crossover_survival_estimate_30}')
print(f'r* for 60d crossover: {crossover_survival_estimate_60}')

r* for 14d crossover: 0.7234973704928684
r* for 30d crossover: 0.6562786975292191
r* for 60d crossover: 0.5302436857223765


In [21]:
print(f'percent high risk at 14d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_14').shape[0]/df.shape[0]}')
print(f'percent high risk at 30d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_30').shape[0]/df.shape[0]}')
print(f'percent high risk at 60d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_60').shape[0]/df.shape[0]}')

percent high risk at 14d crossover: 0.45135135135135135
percent high risk at 30d crossover: 0.3472972972972973
percent high risk at 60d crossover: 0.10810810810810811
